# Targeting Strategy Comparison

This notebook compares multiple targeting strategies under identical campaign constraints.

Strategies evaluated:

- Random targeting
- Predictive outcome targeting
- T-Learner uplift targeting
- X-Learner uplift targeting

Each strategy ranks customers differently.

The objective is to measure profit and incremental response generated by each targeting policy.

In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier

from src.uplift_t_learner import TLearner
from src.uplift_x_learner import XLearner
from src.propensity import train_propensity_model, compute_propensity_scores

In [2]:
df = pd.read_csv("../data/simulated_campaign_data.csv")

## Campaign Economics

Campaign parameters are fixed for all strategies.

Cost per targeted customer: 10  
Revenue per successful conversion: 60  
Targeting budget: 30% of customers

Profit is defined as:

Profit = (Conversions × Margin) − (Targets × Cost)

In [3]:
COST = 10
MARGIN = 60
TARGET_RATIO = 0.3

df_random = df.sample(frac=TARGET_RATIO, random_state=42)

profit_random = df_random["outcome"].sum()*MARGIN - len(df_random)*COST

In [4]:
X = df[["age","income","tenure","usage"]]
y = df["outcome"]

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X,y)

df["naive_score"] = model.predict_proba(X)[:,1]

df_naive = df.sort_values("naive_score",ascending=False)

top_k = int(len(df)*TARGET_RATIO)

targeted_naive = df_naive.head(top_k)

profit_naive = targeted_naive["outcome"].sum()*MARGIN - len(targeted_naive)*COST

In [5]:
tlearner = TLearner()

treatment = df["treatment"]

tlearner.fit(X,treatment,y)

df["uplift_t"] = tlearner.predict_uplift(X)

df_t = df.sort_values("uplift_t",ascending=False)

targeted_t = df_t.head(top_k)

profit_t = targeted_t["outcome"].sum()*MARGIN - len(targeted_t)*COST

In [6]:
model_prop, auc = train_propensity_model(df)

df = compute_propensity_scores(model_prop,df)

propensity = df["propensity_score"]

xlearner = XLearner()

xlearner.fit(X,treatment,y)

df["uplift_x"] = xlearner.predict_uplift(X,propensity)

df_x = df.sort_values("uplift_x",ascending=False)

targeted_x = df_x.head(top_k)

profit_x = targeted_x["outcome"].sum()*MARGIN - len(targeted_x)*COST

In [7]:
results = pd.DataFrame({
"Strategy":["Random","Predictive","T-Learner","X-Learner"],
"Profit":[profit_random,profit_naive,profit_t,profit_x]
})

results

,Strategy,Profit
0,Random,32040
1,Predictive,150000
2,T-Learner,117780
3,X-Learner,77640


## Interpretation

Random targeting provides the baseline performance.

Predictive targeting improves ranking by identifying customers with high purchase probability, but does not isolate treatment effect.

Uplift models explicitly estimate incremental response.

If T-Learner and X-Learner produce higher profit, causal targeting yields measurable economic benefit.

## Result Interpretation

The predictive model achieves the highest profit in this simulation.

This occurs because baseline purchase probability is a stronger signal than treatment effect magnitude in the generated dataset.

Customers with high predicted probability are also highly likely to convert when targeted.

In environments where treatment effect is small relative to baseline conversion probability, predictive targeting may outperform uplift-based strategies.

This highlights an important property of uplift modeling:

Uplift methods provide the greatest advantage when treatment effect heterogeneity is large relative to baseline outcome probability.

In practice, uplift modeling is most valuable in persuasion-sensitive domains such as marketing campaigns, retention incentives, or targeted promotions.